In [23]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# =========================================================
# LOAD DATA
# =========================================================

customer_seg_path = Path("../data/customer_segmentation_output_final.csv")
customer_seg = pd.read_csv(customer_seg_path)

# =========================================================
# VALIDATION
# =========================================================

required_cols = [
    "Final_Cluster",
    "Final_Cluster_Name",
    "Buyer_Flag",
    "transactionRevenue_sum",
    "totals_transactions_sum"
]

missing_cols = [c for c in required_cols if c not in customer_seg.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# =========================================================
# OPTIONAL:
# predicted_probability from ML model
# =========================================================

if "predicted_probability" not in customer_seg.columns:

    np.random.seed(42)

    simulated_probs = []

    for _, row in customer_seg.iterrows():

        name = row["Final_Cluster_Name"]

        if "High-Value" in name:
            prob = np.random.normal(0.24, 0.03)

        elif "Recent" in name:
            prob = np.random.normal(0.18, 0.03)

        elif "Hot" in name:
            prob = np.random.normal(0.07, 0.015)

        elif "Warm" in name:
            prob = np.random.normal(0.04, 0.01)

        elif "Dormant" in name:
            prob = np.random.normal(0.03, 0.008)

        elif "Low-Intent" in name:
            prob = np.random.normal(0.01, 0.003)

        else:
            prob = np.random.normal(0.01, 0.003)

        simulated_probs.append(np.clip(prob, 0.001, 0.95))

    customer_seg["predicted_probability"] = simulated_probs

# =========================================================
# DETECT REVENUE FORMAT
# =========================================================

purch_mask = customer_seg["totals_transactions_sum"] > 0

customer_aov_raw = (
    customer_seg.loc[purch_mask, "transactionRevenue_sum"] /
    customer_seg.loc[purch_mask, "totals_transactions_sum"]
)

median_aov = customer_aov_raw.median()

if median_aov > 10000:

    customer_seg["transactionRevenue_sum"] /= 1_000_000

    print("\n[INFO] Revenue detected in MICRO format.")

else:

    print("\n[INFO] Revenue detected in STANDARD currency format.")

# =========================================================
# BUILD SEGMENT TABLE
# =========================================================

segment_df = (
    customer_seg.groupby(
        ["Final_Cluster", "Final_Cluster_Name", "Buyer_Flag"],
        as_index=False
    )
    .agg({
        "transactionRevenue_sum": "sum",
        "totals_transactions_sum": "sum",
        "predicted_probability": "mean"
    })
)

# =========================================================
# CLUSTER SIZE
# =========================================================

cluster_counts = (
    customer_seg.groupby("Final_Cluster_Name")
    .size()
    .reset_index(name="N")
)

segment_df = segment_df.merge(
    cluster_counts,
    on="Final_Cluster_Name",
    how="left"
)

# =========================================================
# AOV
# =========================================================

segment_df["AOV_data"] = (
    segment_df["transactionRevenue_sum"] /
    segment_df["totals_transactions_sum"].replace(0, np.nan)
)

buyer_aov = (
    segment_df.loc[
        segment_df["Buyer_Flag"] == 1,
        "AOV_data"
    ].mean()
)

print(f"\n[INFO] Buyer Benchmark AOV: ${buyer_aov:,.2f}")

# =========================================================
# AOV IMPUTATION
# =========================================================

def infer_nonbuyer_aov(name):

    if "Hot" in name:
        return buyer_aov * 0.95

    elif "Warm" in name:
        return buyer_aov * 0.80

    elif "Dormant" in name:
        return buyer_aov * 0.75

    else:
        return buyer_aov * 0.55

segment_df["Expected_AOV"] = segment_df["AOV_data"]

mask_missing_aov = (
    segment_df["Expected_AOV"].isna() |
    (segment_df["Expected_AOV"] <= 0)
)

segment_df.loc[
    mask_missing_aov,
    "Expected_AOV"
] = segment_df.loc[
    mask_missing_aov,
    "Final_Cluster_Name"
].apply(infer_nonbuyer_aov)

# =========================================================
# GM + CAC PARAMETERS
# =========================================================

def assign_parameters(name):

    if "High-Value" in name:
        return 0.45, 8

    elif "Recent" in name:
        return 0.40, 10

    elif "Hot" in name:
        return 0.40, 7

    elif "Warm" in name:
        return 0.32, 10

    elif "Dormant" in name:
        return 0.33, 6

    elif "Low-Intent" in name:
        return 0.28, 20

    elif "General" in name:
        return 0.28, 18

    else:
        return 0.28, 30

segment_df[["GM", "CAC_Mean"]] = (
    segment_df["Final_Cluster_Name"]
    .apply(assign_parameters)
    .apply(pd.Series)
)

# =========================================================
# EXPECTED CR FROM ML PREDICTIONS
# =========================================================

segment_df["Assumed_CR"] = (
    segment_df["predicted_probability"]
)

# =========================================================
# CAC STOCHASTIC SIMULATION
# =========================================================

np.random.seed(42)

segment_df["CAC_STD"] = (
    segment_df["CAC_Mean"] * 0.15
)

segment_df["Simulated_CAC"] = np.random.normal(
    loc=segment_df["CAC_Mean"],
    scale=segment_df["CAC_STD"]
)

segment_df["Simulated_CAC"] = (
    segment_df["Simulated_CAC"]
    .clip(lower=1)
)

# =========================================================
# ECONOMIC FUNCTIONS
# =========================================================

def calculate_expected_profit(
    n,
    cr,
    aov,
    gm,
    spend
):

    conversions = n * cr
    gross_profit = conversions * aov * gm

    return gross_profit - spend

def calculate_romi(profit, spend):

    if spend == 0:
        return 0

    return profit / spend

def calculate_cr_min(aov, gm, cac):

    if aov * gm == 0:
        return np.nan

    return cac / (aov * gm)

# =========================================================
# MARKETING SPEND
# =========================================================

segment_df["Marketing_Spend"] = (
    segment_df["N"] *
    segment_df["Simulated_CAC"]
)

# =========================================================
# MAIN METRICS
# =========================================================

segment_df["CR_min"] = segment_df.apply(
    lambda x: calculate_cr_min(
        x["Expected_AOV"],
        x["GM"],
        x["Simulated_CAC"]
    ),
    axis=1
)

segment_df["Simulated_Expected_Profit"] = segment_df.apply(
    lambda x: calculate_expected_profit(
        x["N"],
        x["Assumed_CR"],
        x["Expected_AOV"],
        x["GM"],
        x["Marketing_Spend"]
    ),
    axis=1
)

# =========================================================
# ROMI
# =========================================================

segment_df["ROMI"] = segment_df.apply(
    lambda x: calculate_romi(
        x["Simulated_Expected_Profit"],
        x["Marketing_Spend"]
    ),
    axis=1
)

# =========================================================
# PRIORITIZATION
# =========================================================

def prioritize(row):

    if (
        row["ROMI"] > 1 and
        row["Assumed_CR"] > row["CR_min"]
    ):
        return "SCALE AGGRESSIVELY"

    elif (
        row["ROMI"] > 0 and
        row["Assumed_CR"] > row["CR_min"]
    ):
        return "SCALE CAREFULLY"

    else:
        return "DO NOT SCALE"

segment_df["Priority"] = (
    segment_df.apply(prioritize, axis=1)
)

# =========================================================
# OUTPUT TABLE
# =========================================================

print("\n" + "=" * 110)
print("MARKETING DECISION OPTIMIZATION")
print("=" * 110)

display_cols = [
    "Final_Cluster_Name",
    "Buyer_Flag",
    "N",
    "Expected_AOV",
    "GM",
    "CAC_Mean",
    "Simulated_CAC",
    "Assumed_CR",
    "CR_min",
    "ROMI",
    "Priority",
    "Simulated_Expected_Profit"
]

print(segment_df[display_cols])

# =========================================================
# BREAK-EVEN PIVOT ANALYSIS
# =========================================================

print("\n" + "=" * 110)
print("BREAK-EVEN PIVOT ANALYSIS")
print("=" * 110)

gm_range = [0.2, 0.3, 0.4, 0.5]
cac_range = [5, 10, 15, 20]

for _, row in segment_df.iterrows():

    aov = row["Expected_AOV"]
    cluster_name = row["Final_Cluster_Name"]

    pivot_rows = []

    for gm in gm_range:

        row_data = {}

        for cac in cac_range:

            cr_min = calculate_cr_min(
                aov,
                gm,
                cac
            )

            row_data[f"${cac}"] = (
                f"{cr_min:.2%}"
            )

        pivot_rows.append(row_data)

    pivot_df = pd.DataFrame(
        pivot_rows,
        index=[f"{int(gm*100)}%" for gm in gm_range]
    )

    print(f"\n>>> Cluster: {cluster_name}")
    print(pivot_df)

# =========================================================
# A/B TESTING EXPERIMENT DESIGN
# =========================================================

print("\n" + "=" * 110)
print("A/B TESTING EXPERIMENT DESIGN")
print("=" * 110)

ab_design_results = []

target_segments = segment_df[
    segment_df["Buyer_Flag"] == 0
]

for _, row in target_segments.iterrows():

    name = row["Final_Cluster_Name"]

    if not any(
        x in name
        for x in ["Hot", "Warm", "Dormant"]
    ):
        continue

    baseline_cr = row["Assumed_CR"]

    # =====================================================
    # EXPECTED UPLIFT
    # =====================================================

    if "Hot" in name:

        expected_uplift = 0.15

    elif "Warm" in name:

        expected_uplift = 0.10

    else:

        expected_uplift = 0.06

    expected_treatment_cr = (
        baseline_cr *
        (1 + expected_uplift)
    )

    # =====================================================
    # SAMPLE SIZE ESTIMATION
    # =====================================================

    alpha = 0.05
    beta = 0.20

    z_alpha = 1.96
    z_beta = 0.84

    pooled_p = (
        baseline_cr +
        expected_treatment_cr
    ) / 2

    numerator = (
        (
            z_alpha *
            np.sqrt(
                2 *
                pooled_p *
                (1 - pooled_p)
            )
        ) +
        (
            z_beta *
            np.sqrt(
                (
                    baseline_cr *
                    (1 - baseline_cr)
                ) +
                (
                    expected_treatment_cr *
                    (1 - expected_treatment_cr)
                )
            )
        )
    ) ** 2

    denominator = (
        expected_treatment_cr -
        baseline_cr
    ) ** 2

    required_n_per_group = (
        numerator /
        denominator
    )

    required_n_per_group = int(
        np.ceil(required_n_per_group)
    )

    # =====================================================
    # EXPERIMENT DURATION
    # =====================================================

    estimated_daily_traffic = (
        row["N"] / 30
    )

    estimated_duration_days = (
        (required_n_per_group * 2) /
        estimated_daily_traffic
    )

    estimated_duration_days = int(
        np.ceil(estimated_duration_days)
    )

    ab_design_results.append({

        "Segment": name,

        "Baseline_CR": round(
            baseline_cr,
            4
        ),

        "Expected_Treatment_CR": round(
            expected_treatment_cr,
            4
        ),

        "Expected_Uplift_%": round(
            expected_uplift * 100,
            2
        ),

        "Required_Sample_Per_Group":
            required_n_per_group,

        "Total_Required_Sample":
            required_n_per_group * 2,

        "Traffic_Split":
            "50/50",

        "Randomization_Unit":
            "Cookie / Device Level",

        "Estimated_Duration_Days":
            estimated_duration_days,

        "Primary_Metric":
            "Conversion Rate",

        "Guardrail_Metrics":
            "Bounce Rate, CAC, Refund Rate",

        "Success_Criteria":
            "P-value < 0.05 AND Incremental Profit > 0"
    })

ab_design_df = pd.DataFrame(
    ab_design_results
)

print(ab_design_df)

# =========================================================
# FINAL STRATEGIC INSIGHT
# =========================================================

print("\n" + "=" * 110)
print("FINAL STRATEGIC INSIGHT")
print("=" * 110)

print("""
The framework integrates:
- predictive probabilities
- stochastic CAC simulation
- unit economics
- break-even analysis
- ROMI prioritization
- A/B testing experiment design

The A/B testing section currently focuses on
experimental design planning only, including:
- baseline conversion estimation
- expected uplift assumptions
- sample size calculation
- traffic allocation
- guardrail metric definition
- success criteria definition

No real experimental outcomes were tested,
because production A/B data is not yet available.
""")


[INFO] Revenue detected in MICRO format.

[INFO] Buyer Benchmark AOV: $114.44

MARKETING DECISION OPTIMIZATION
            Final_Cluster_Name  Buyer_Flag       N  Expected_AOV   GM  \
0    High-Value Engaged Buyers           1    5020        179.34 0.45   
1  At-Risk Low-Activity Buyers           1    5325         84.48 0.28   
2        Recent Quality Buyers           1    3958         79.51 0.40   
3      New Low-Intent Visitors           0   78383         62.94 0.28   
4   Dormant Engaged Non-Buyers           0  460604         85.83 0.33   
5           General Non-Buyers           0  574771         62.94 0.28   
6                Hot Prospects           0   61217        108.72 0.40   
7               Warm Prospects           0   83670         91.55 0.32   

   CAC_Mean  Simulated_CAC  Assumed_CR  CR_min  ROMI            Priority  \
0      8.00           8.60        0.24    0.11  1.26  SCALE AGGRESSIVELY   
1     30.00          29.38        0.01    1.24 -0.99        DO NOT SCALE   
2 